In [4]:
import pyodbc
import os
from dotenv import load_dotenv

# Verify odbc SQL drivers
pyodbc.drivers()

['SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
 'ODBC Driver 18 for SQL Server']

In [5]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Load the .csv 
CSV_PATH = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(CSV_PATH)

# The CRM stores TotalCharges as a number: blanks become NULL
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(df.shape)
print("TotalCharges NULL:", df["TotalCharges"].isna().sum())

(7043, 21)
TotalCharges NULL: 11


In [7]:
# Set up the connection to SQL Server (admin login, read from .env)
from sqlalchemy import text

load_dotenv(".env")

conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={os.getenv('DB_SERVER', 'localhost,1433')};"
    "DATABASE=ChurnDB;"
    f"UID={os.getenv('ADMIN_USER')};"
    f"PWD={os.getenv('ADMIN_PASSWORD')};"
    "TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}",
                       fast_executemany=True)


with engine.connect() as conn:
    existing = conn.execute(text("SELECT COUNT(*) FROM dbo.Customers")).scalar()

if existing:
    print(f"dbo.Customers already has {existing} rows: load skipped")
else:
    df.to_sql("Customers", engine, schema="dbo", if_exists="append", index=False)
    print("Loaded:", len(df), "rows")

dbo.Customers already has 7043 rows: load skipped
